# Portfolio Rebalancing Strategies

<div style="background-color: #e3f2fd; border-left: 6px solid #1976d2; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📘 About This Notebook</h4>
This notebook demonstrates dynamic portfolio rebalancing strategies using the core Mean-CVaR optimization module. Rebalancing adapts portfolio weights in response to market changes, aiming to maintain optimal performance while accounting for transaction costs and turnover constraints.
<br><br>
Transaction cost model: <code>transaction_cost = transaction_cost_factor × turnover</code>, where turnover $=\lVert w-w_{pre} \rVert_1$.
<br><br>
We compare two rebalancing triggers:
<ul>
<li><a href="#percentage-change"><strong>Percentage Change</strong></a> — reoptimize when portfolio value drops below a threshold</li>
<li><a href="#drift-from-optimal"><strong>Drift from Optimal</strong></a> — reoptimize when current weights deviate from optimal by a specified norm</li>
</ul>
Both strategies are benchmarked against a buy-and-hold baseline using GPU (cuOpt) vs. CPU solvers.
</div>


In [ ]:
# Runs first: CUDA ignores CUDA_VISIBLE_DEVICES once cvxpy, cuOpt, or cuML is imported.
# Set it in ../.env (see .env.sample), GPU 0 when unset; restart the kernel to change it.
from portfolio_optimization.gpu import apply_gpu_selection

print(f"CUDA_VISIBLE_DEVICES={apply_gpu_selection('../.env')!r}")

In [ ]:
import cvxpy as cp
import pandas as pd

from portfolio_optimization import cvar_optimizer, cvar_utils, rebalance, utils
from portfolio_optimization.cvar_parameters import CvarParameters
from portfolio_optimization.gpu import run_on_gpus
from portfolio_optimization.settings import (
    ApiSettings,
    JointFMSettings,
    # KDESettings,
    ReturnsComputeSettings,
    ScenarioGenerationSettings,
)

In [ ]:
# Global data settings.
# Define dataset paths and common parameters.
sp500_dataset_name = "sp500"
sp500_dataset_directory = f"../data/stock_data/{sp500_dataset_name}.csv"

# JointFM forecasts all assets jointly in one request, so the universe must fit
# the served model's series limit (health().data_generation.max_series).
# A seeded random sample keeps the universe independent of the CSV's alphabetical column order.
max_series = 100
universe_seed = 0
rebal_dataset_directory = f"../data/stock_data/{sp500_dataset_name}_random{max_series}_seed{universe_seed}.csv"
sp500_prices = pd.read_csv(sp500_dataset_directory, index_col=0, parse_dates=True)
rebal_prices = sp500_prices.sample(n=max_series, axis=1, random_state=universe_seed)
rebal_prices.to_csv(rebal_dataset_directory)

# Define the settings for computing returns and scenario generation
rebal_returns_compute_settings = ReturnsComputeSettings(return_type="LOG", freq=1)
rebal_scenario_generation_settings = ScenarioGenerationSettings(
    num_scen=10000,  # Number of return scenarios to simulate
    fit_type="jointfm",
    # The client's .env lives in the repository root, one level up from here.
    jointfm_settings=JointFMSettings(dotenv_path="../.env"),
    # kde_settings=KDESettings(bandwidth=0.01, kernel="gaussian", device="GPU"),
    verbose=False,
)

In [ ]:
# Define CVaR parameters for rebalancing with transaction costs and turnover constraints.
rebal_tc_cvar_params = CvarParameters(
    w_min=-0.3,
    w_max=0.8,
    c_min=0.1,
    c_max=0.4,
    L_tar=1.6,
    T_tar=0.5,  # Turnover constraint to limit trading activity.
    cvar_limit=None,
    risk_aversion=1,
    confidence=0.95,
)

# One absolute risk aversion for every look-back window. By default the optimizer
# multiplies risk_aversion by the window's best single-asset return/CVaR ratio,
# which ties the risk preference to recent drift and fails in a window where every
# asset lost money. Take that ratio once from the history before trading starts
# (later data would leak into the backtest) and switch the per-window scaling off.
risk_aversion_calibration_end = "2022-07-01"  # Trading start of both strategies.
calibration_returns_dict = cvar_utils.generate_cvar_data(
    utils.calculate_returns(
        rebal_prices,
        {
            "name": "risk-aversion calibration",
            "range": (
                rebal_prices.index[0].strftime("%Y-%m-%d"),
                risk_aversion_calibration_end,
            ),
        },
        rebal_returns_compute_settings,
    ),
    ScenarioGenerationSettings(fit_type="no_fit"),
)
calibration_problem = cvar_optimizer.CVaR(
    returns_dict=calibration_returns_dict, cvar_params=rebal_tc_cvar_params
)
rebal_tc_cvar_params.update_risk_aversion(calibration_problem.params.risk_aversion)
rebal_api_settings = ApiSettings(scale_risk_aversion=False)
print(f"Absolute risk aversion: {rebal_tc_cvar_params.risk_aversion:.4g}")

## Strategy 1: Percentage Change Rebalancing
<a id='percentage-change'></a>

<div style="background-color: #f3e5f5; border-left: 6px solid #9c27b0; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💡 Trigger Conditions</h4>
Re-optimization is triggered if either condition holds:
<ul>
<li>The percent change from the start to the end of the testing period drops below <code>pct_change_tolerance</code></li>
<li>The cumulative percent change since the <em>last positive</em> backtest period drops below <code>pct_change_tolerance</code></li>
</ul>
Example setup: 21-day (one-month) look-forward window and 252-day (one-year) look-back window for rolling optimization, with cuOpt as the GPU solver.
</div> 


In [ ]:
# Trading period and backtesting windows.
selected_rebal_scenario_name = "rebalancing_trading_period"
rebal_trading_start_date, rebal_trading_end_date = "2022-07-01", "2024-01-01"

rebal_look_back_window = 252  # Historical period for optimization (trading days).
rebal_look_forward_window = 21  # Out-of-sample testing period (trading days).

# Transaction cost parameters
transaction_cost_factor = 0.001

# Reoptimization trigger: percentage change threshold.
percent_change_tolerance = -0.005  # Reoptimize if portfolio value drops by 0.5%.
pct_change_re_optimize_criteria = {
    "type": "pct_change",
    "threshold": percent_change_tolerance,
}

# GPU solver configuration.
solver_settings = {"solver": cp.CUOPT, "verbose": False, "solver_method": "PDLP"}

# Portfolio rebalancing with percentage change trigger, run below with the other strategy.
pct_change_run = {
    "rebalancer_kwargs": dict(
        dataset_directory=rebal_dataset_directory,
        returns_compute_settings=rebal_returns_compute_settings,
        scenario_generation_settings=rebal_scenario_generation_settings,
        trading_start=rebal_trading_start_date,
        trading_end=rebal_trading_end_date,
        look_forward_window=rebal_look_forward_window,
        look_back_window=rebal_look_back_window,
        cvar_params=rebal_tc_cvar_params,
        solver_settings=solver_settings,
        re_optimize_criteria=pct_change_re_optimize_criteria,  # specify the re-optimization criteria
        print_opt_result=False,
        api_settings=rebal_api_settings,
    ),
    "re_optimize_kwargs": dict(transaction_cost_factor=transaction_cost_factor),
}

## Strategy 2: Drift from Optimal Portfolio
<a id='drift-from-optimal'></a>

<div style="background-color: #f3e5f5; border-left: 6px solid #9c27b0; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💡 Drift-Based Trigger</h4>
Over time the portfolio composition drifts from the original optimized allocation. Re-optimization is triggered when:
$$\lVert w_{cur} - w_{opt} \rVert_* > tol,$$
where:
<ul>
<li>$w_{cur}$ = Current portfolio weights</li>
<li>$w_{opt}$ = Optimal portfolio weights</li>
<li>$tol$ = <code>drift_tolerance</code></li>
<li>Norm $*$: 1-norm (absolute differences) or 2-norm (Euclidean distance)</li>
</ul>
</div>

In [ ]:
# --- Select Scenario for Rebalancing ---
selected_rebal_scenario_name = "rebalancing_trading_period"
rebal_trading_start_date, rebal_trading_end_date = "2022-07-01", "2024-05-01"

# Look-back and look-forward windows for backtesting
rebal_look_back_window = 252  # Historical period used for optimization (trading days)
rebal_look_forward_window = 42  # Testing period (out-of-sample performance)

# Define drift tolerance threshold
drift_rebal_tolerance = (
    0.05  # Rebalance if weight deviation (L2 norm) exceeds tolerance
)

# Set re-optimization criteria for drift
drift_re_optimize_criteria = {
    "type": "drift_from_optimal",
    "threshold": drift_rebal_tolerance,
    "norm": 1,  # Using L2 norm
}

# GPU solver
solver_settings = {"solver": cp.CUOPT, "verbose": False, "solver_method": "PDLP"}

# Portfolio rebalancing with drift trigger, run below with the other strategy.
drift_run = {
    "rebalancer_kwargs": dict(
        dataset_directory=rebal_dataset_directory,
        returns_compute_settings=rebal_returns_compute_settings,
        scenario_generation_settings=rebal_scenario_generation_settings,
        trading_start=rebal_trading_start_date,
        trading_end=rebal_trading_end_date,
        look_forward_window=rebal_look_forward_window,
        look_back_window=rebal_look_back_window,
        cvar_params=rebal_tc_cvar_params,
        solver_settings=solver_settings,
        re_optimize_criteria=drift_re_optimize_criteria,
        print_opt_result=False,
        api_settings=rebal_api_settings,
    ),
    "re_optimize_kwargs": dict(),
}

## Running the Strategies
<a id='running-the-strategies'></a>

<div style="background-color: #f3e5f5; border-left: 6px solid #9c27b0; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💡 One Strategy per GPU</h4>
The strategies are independent, so <code>run_on_gpus</code> runs each one in its own worker process on its own GPU when <code>CUDA_VISIBLE_DEVICES</code> lists several, and in sequence in this kernel when it lists one. A worker's printed analysis summary goes to the Jupyter server log instead of this notebook, so the summary is printed again below, and the plots are drawn here from the returned results.
</div>

In [ ]:
# Run both strategies, one per GPU listed in CUDA_VISIBLE_DEVICES.
(
    (pct_change_rebalancing_obj, pct_change_result),
    (drift_rebalancing_obj, drift_result),
) = run_on_gpus(rebalance.run_rebalancing, [pct_change_run, drift_run])
pct_change_results_df, pct_change_re_optimize_dates, _ = pct_change_result
drift_results_df, drift_re_optimize_dates, _ = drift_result

# Summarize and plot each strategy against its buy-and-hold baseline.
for rebalancing_obj, (results_df, re_optimize_dates, cumulative_portfolio_value) in [
    (pct_change_rebalancing_obj, pct_change_result),
    (drift_rebalancing_obj, drift_result),
]:
    total_return = (
        cumulative_portfolio_value.iloc[-1] / cumulative_portfolio_value.iloc[0] - 1
    ) * 100
    print(
        f"{rebalancing_obj.re_optimize_type}: {len(re_optimize_dates)} rebalancing "
        f"events, total return {total_return:+.2f}%"
    )
    rebalancing_obj.plot_results(
        results_df,
        re_optimize_dates,
        cumulative_portfolio_value,
        save_plot=True,
        results_dir="../results/rebalancing_strategies",
    )

<div style="background-color: #e8f5e9; border-left: 6px solid #388e3c; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🚀 Why GPU Acceleration Matters Here</h4>
Portfolio optimization is a repeated core task rather than a one-time effort — the acceleration cuOpt delivers for a single solve is magnified in active trading environments. More sophisticated strategies with additional hyperparameters require countless optimization iterations. These examples highlight GPU-accelerated performance; finding optimal strategies for deployment requires many more iterations.
</div> 


SPDX-FileCopyrightText: Copyright (c) 2023-2025 NVIDIA CORPORATION & AFFILIATES. All rights reserved. 

SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0. Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.